# CIFAR-100 coarse-label audit (CPU)
Run all cells in a CPU runtime. No model, GPU, TensorFlow, or PyTorch is needed; only NumPy and Python's standard library. This notebook is standalone and downloads the original Python archive unless a verified copy is already cached. Reports go to `./cifar100_audit` by default. No Drive mount is needed. To reuse an existing archive, set `CIFAR100_ARCHIVE` to its local path before running, or edit the configuration cell.

The targets are the 20 official coarse IDs. Fine IDs are only for auditing, stratifying the official training partition, and evaluating coarse predictions by true subtype—never classifier fitting. The official test split is not used to select validation examples.

In [ ]:
import csv
import hashlib
import json
import os
from pathlib import Path
import pickle
import platform
import tarfile
from tempfile import NamedTemporaryFile
from time import monotonic
from urllib.request import Request, urlopen

import numpy as np

OUT = Path(os.environ.get('CIFAR100_AUDIT_OUT', 'cifar100_audit')).resolve()
ARCHIVE = 'cifar-100-python.tar.gz'
SOURCES = [
    'https://cave.cs.toronto.edu/kriz/' + ARCHIVE,
    'https://storage.googleapis.com/tensorflow/tf-keras-datasets/' + ARCHIVE,
    'https://data.brainchip.com/dataset-mirror/cifar100/' + ARCHIVE,
]
REFERENCES = {
    'publisher': 'https://www.cs.toronto.edu/~kriz/cifar.html',
    'keras': 'https://github.com/keras-team/keras/blob/master/keras/src/datasets/cifar100.py',
    'torchvision': 'https://github.com/pytorch/vision/blob/main/torchvision/datasets/cifar.py',
}
ARCHIVE_HASH = {
    'md5': 'eb9058c3a382ffc7106e4002c42a8d85',
    'sha256': '85cd44d02ba6437773c5bbd22e183051d648de2e7d6b014e1ef29b855ba677a7',
}
MEMBER_MD5 = {
    'train': '16019d7e3df5f24257cddd939b257f8d',
    'test': 'f0ef6b0ae62326f3e7ffdfab6717acfc',
    'meta': '7973b15100ade9c7d40fb424638fde48',
}
SEED = 42
VAL_PER_FINE = 50

## 1. Authenticate and read the archive
The archive must match both the publisher's MD5 and the SHA-256 pinned by Keras. Verify each member against TorchVision before unpickling. Read named regular files directly from the archive; do not extract paths to disk.

In [ ]:
def archive_hashes(path):
    counters = {name: hashlib.new(name) for name in ARCHIVE_HASH}
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            for counter in counters.values():
                counter.update(chunk)
    return {name: counter.hexdigest() for name, counter in counters.items()}


def acquire():
    override = os.environ.get('CIFAR100_ARCHIVE')
    if override:
        path = Path(override).expanduser()
        if archive_hashes(path) != ARCHIVE_HASH:
            raise ValueError(f'Archive checksum mismatch: {path}')
        return path, os.environ.get('CIFAR100_ARCHIVE_ORIGIN', 'provided local archive')
    path = OUT / '.cache' / ARCHIVE
    origin_file = path.with_suffix('.origin.txt')
    if path.is_file() and archive_hashes(path) == ARCHIVE_HASH:
        return path, origin_file.read_text().strip() if origin_file.exists() else 'verified cache (origin not recorded)'
    path.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in SOURCES:
        tmp = None
        try:
            print(f'Trying {url}', flush=True)
            with NamedTemporaryFile(dir=path.parent, suffix='.part', delete=False) as f:
                tmp = Path(f.name)
                deadline = monotonic() + 180
                with urlopen(Request(url, headers={'User-Agent': 'CIFAR100-audit/1.0'}), timeout=15) as response:
                    while chunk := response.read1(1024 * 1024):
                        if monotonic() > deadline:
                            raise TimeoutError('download exceeded 180 seconds')
                        f.write(chunk)
            if archive_hashes(tmp) != ARCHIVE_HASH:
                raise ValueError('archive checksum mismatch (possibly a partial download)')
            tmp.replace(path)
            origin_file.write_text(url + chr(10))
            return path, url
        except (OSError, ValueError) as exc:
            errors.append(f'{url}: {exc}')
            print(errors[-1])
        finally:
            if tmp is not None:
                tmp.unlink(missing_ok=True)
    raise RuntimeError('No verified CIFAR-100 archive available: ' + '; '.join(errors))


def read_members(path):
    result = {}
    observed = {}
    with tarfile.open(path, 'r:gz') as tar:
        for member in tar:
            folder, _, name = member.name.partition('/')
            if folder != 'cifar-100-python' or name not in MEMBER_MD5:
                continue
            if not member.isfile() or name in result:
                raise ValueError(f'Unexpected archive member: {member.name}')
            with tar.extractfile(member) as f:
                raw = f.read()
            observed[name] = hashlib.md5(raw).hexdigest()
            if observed[name] != MEMBER_MD5[name]:
                raise ValueError(f'Member checksum mismatch: {member.name}')
            payload = pickle.loads(raw, encoding='latin1')
            if name != 'meta':
                n = 50_000 if name == 'train' else 10_000
                assert payload['data'].shape == (n, 3072) and payload['data'].dtype == np.uint8
                assert len(payload['filenames']) == n
                payload = {key: np.asarray(payload[key]) for key in ('fine_labels', 'coarse_labels')}
            result[name] = payload
            del raw
    if observed != MEMBER_MD5:
        raise ValueError('Archive is missing one or more verified members')
    return result, observed

archive_path, archive_origin = acquire()
arrays, member_hashes = read_members(archive_path)
print(f'Archive origin: {archive_origin}\nVerified archive MD5/SHA-256 and all 3 member MD5s.')

## 2. Audit the label pairs and distributions
Read class names in official ID order from `meta`. Count the observed `(fine, coarse)` label pairs independently in both official partitions before deriving the common fine-to-coarse mapping.

In [ ]:
fine_names = np.asarray(arrays['meta']['fine_label_names'])
coarse_names = np.asarray(arrays['meta']['coarse_label_names'])
for names, n in [(fine_names, 100), (coarse_names, 20)]:
    assert names.shape == (n,) and names.dtype.kind == 'U'
    assert len(set(names)) == n and all(name.strip() for name in names)


def label_pairs(batch, n):
    fine, coarse = batch['fine_labels'], batch['coarse_labels']
    for labels, upper in [(fine, 100), (coarse, 20)]:
        assert labels.shape == (n,) and labels.dtype.kind in 'iu'
        assert np.array_equal(np.unique(labels), np.arange(upper))
    pairs = np.zeros((100, 20), dtype=np.int64)
    np.add.at(pairs, (fine, coarse), 1)
    assert np.all(np.count_nonzero(pairs, axis=1) == 1)
    assert np.all(np.count_nonzero(pairs, axis=0) == 5)
    return fine, coarse, pairs

train_fine, train_coarse, train_pairs = label_pairs(arrays['train'], 50_000)
test_fine, test_coarse, test_pairs = label_pairs(arrays['test'], 10_000)
fine_to_coarse = train_pairs.argmax(axis=1)
assert np.array_equal(fine_to_coarse, test_pairs.argmax(axis=1))
assert np.array_equal(fine_to_coarse[train_fine], train_coarse)
assert np.array_equal(fine_to_coarse[test_fine], test_coarse)
for pairs, fine_n, coarse_n in [(train_pairs, 500, 2500), (test_pairs, 100, 500)]:
    assert np.all(pairs.sum(axis=1) == fine_n) and np.all(pairs.sum(axis=0) == coarse_n)
print('60,000 label pairs verified: 100 fine classes, 20 coarse classes, exactly 5 subtypes per superclass.')

## 3. Split the official training rows
Within each fine class, rank original row IDs by SHA-256 of the ASCII string `cifar100-val-v1:<seed>:<row_id>`, breaking ties by numeric row ID. Take the first 50 for validation and the rest for fit. Return each partition in ascending original row order. SHA-256 ranking avoids RNG-version dependencies. Check same-seed repeatability, changed-seed sensitivity, exact membership, and both label balances. Test row IDs have a separate namespace; equal numeric indices across files are not the same row.

In [ ]:
def split_train(seed):
    holdout = []
    for fid in range(100):
        rows = np.flatnonzero(train_fine == fid).tolist()
        rows.sort(key=lambda i: (hashlib.sha256(f'cifar100-val-v1:{seed}:{i}'.encode('ascii')).digest(), i))
        holdout.extend(rows[:VAL_PER_FINE])
    validation = np.asarray(sorted(holdout), dtype=np.int64)
    keep = np.ones(len(train_fine), dtype=bool)
    keep[validation] = False
    return np.flatnonzero(keep), validation


def index_sha256(rows):
    return hashlib.sha256(np.asarray(rows, dtype='<i8').tobytes()).hexdigest()

fit_indices, validation_indices = split_train(SEED)
assert len(fit_indices) == 45_000 and len(validation_indices) == 5_000
assert all(np.all(np.diff(x) > 0) for x in (fit_indices, validation_indices))
assert np.intersect1d(fit_indices, validation_indices).size == 0
assert np.array_equal(np.union1d(fit_indices, validation_indices), np.arange(50_000))
assert all(np.array_equal(a, b) for a, b in zip(split_train(SEED), (fit_indices, validation_indices)))
assert not np.array_equal(split_train(SEED + 1)[1], validation_indices)

labels = {
    'official_train': (train_fine, train_coarse),
    'fit': (train_fine[fit_indices], train_coarse[fit_indices]),
    'validation': (train_fine[validation_indices], train_coarse[validation_indices]),
    'official_test': (test_fine, test_coarse),
}
counts = {k: (np.bincount(f, minlength=100), np.bincount(c, minlength=20))
          for k, (f, c) in labels.items()}
for key, fine_n, coarse_n in [('official_train', 500, 2500), ('fit', 450, 2250),
                              ('validation', 50, 250), ('official_test', 100, 500)]:
    fine_count, coarse_count = counts[key]
    assert np.all(fine_count == fine_n) and np.all(coarse_count == coarse_n)
    print(f'{key:>14}: {fine_count.sum():,} rows; {fine_n} per fine, {coarse_n} per coarse')
print('Validation SHA-256:', index_sha256(validation_indices))

## 4. Save the audit manifest and tables
The compressed NPZ preserves paired labels in original archive order, class names, the mapping, and the fit/validation indices. The CSV files contain all rows of the label hierarchy and distribution tables.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT / 'audit_manifest.npz'
np.savez_compressed(MANIFEST, train_fine=train_fine, train_coarse=train_coarse,
                    test_fine=test_fine, test_coarse=test_coarse,
                    fit_indices=fit_indices, validation_indices=validation_indices,
                    fine_to_coarse=fine_to_coarse, fine_names=fine_names, coarse_names=coarse_names)


def write_csv(name, columns, rows):
    with (OUT / name).open('w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)

write_csv('fine_counts.csv', ['fine_id', 'fine_name', 'coarse_id', 'coarse_name', *counts],
          ([i, name, fine_to_coarse[i], coarse_names[fine_to_coarse[i]], *[f[i] for f, c in counts.values()]]
           for i, name in enumerate(fine_names)))
write_csv('coarse_counts.csv', ['coarse_id', 'coarse_name', 'fine_subtypes', *counts],
          ([i, name, ';'.join(fine_names[fine_to_coarse == i]), *[c[i] for f, c in counts.values()]]
           for i, name in enumerate(coarse_names)))
print('Manifest and distributions:', OUT)

## 5. Evaluate later coarse predictions by subtype
Pass original row IDs in the same order as your predictions. The helper supports a length-N integer vector or an N×20 score matrix in official coarse ID order. It reports overall coarse accuracy, macro-average subtype accuracy, and one row per **true fine subtype**, with support, correct count, accuracy, and wrong predicted coarse IDs. Use `manifest=MANIFEST` in the notebook. The following checks use constructed predictions only, **not** a classifier or benchmark result.

In [ ]:
"""CPU-only coarse-classifier diagnostics by true CIFAR-100 fine subtype."""
from pathlib import Path

import numpy as np


def _integer_vector(values, name, length, upper):
    values = np.asarray(values)
    if values.shape != (length,) or values.dtype.kind not in 'iu':
        raise ValueError(f'{name} must be a 1-D integer array of length {length}')
    if np.any(values < 0) or np.any(values >= upper):
        raise ValueError(f'{name} values must be in [0, {upper})')
    return values.astype(np.int64, copy=False)


def evaluate_subtypes(predictions, *, split, row_ids, manifest=None):
    """Evaluate IDs [N] or finite scores [N,20] in official coarse-ID order.

    Pass one original 0-based row ID per prediction, in prediction order.
    For validation use official *train* row IDs; for test use official test row IDs.
    Requires exactly the full named split, but supports arbitrary row order.
    Accuracy is the fraction predicting the subtype's true *coarse* parent.
    Tied scores select the lowest coarse ID. No fine predictions are required.
    """
    if split not in ('validation', 'test'):
        raise ValueError('split must be validation or test')
    if manifest is None:
        base = Path(__file__).resolve().parent if '__file__' in globals() else OUT
        manifest = base / 'audit_manifest.npz'
    with np.load(manifest, allow_pickle=False) as data:
        source = 'train' if split == 'validation' else 'test'
        all_fine, all_coarse = data[f'{source}_fine'], data[f'{source}_coarse']
        expected = data['validation_indices'] if split == 'validation' else np.arange(len(all_fine))
        names, parents, coarse_names = (data[k] for k in ('fine_names', 'fine_to_coarse', 'coarse_names'))
    n = len(expected)
    row_ids = _integer_vector(row_ids, 'row_ids', n, len(all_fine))
    if not np.array_equal(np.sort(row_ids), expected):
        raise ValueError(f'row_ids must cover {split} exactly once (no duplicates or foreign rows)')
    pred = np.asarray(predictions)
    if pred.ndim == 2:
        if pred.shape != (n, 20) or pred.dtype.kind not in 'fiu' or not np.isfinite(pred).all():
            raise ValueError(f'scores must be a finite numeric ({n}, 20) array')
        pred = pred.argmax(axis=1)
    pred = _integer_vector(pred, 'predicted coarse IDs', n, 20)
    fine, coarse = all_fine[row_ids], all_coarse[row_ids]
    matrix = np.zeros((100, 20), dtype=np.int64)
    np.add.at(matrix, (fine, pred), 1)
    rows = []
    for fid, parent in enumerate(parents):
        counts = matrix[fid]
        support, correct = int(counts.sum()), int(counts[parent])
        rows.append(dict(fine_id=fid, fine_name=str(names[fid]), coarse_id=int(parent),
                         coarse_name=str(coarse_names[parent]), support=support, correct=correct,
                         accuracy=correct / support if support else None,
                         wrong_coarse_counts={int(cid): int(counts[cid]) for cid in np.flatnonzero(counts) if cid != parent}))
    return dict(split=split, n=n, correct=int(np.count_nonzero(pred == coarse)),
                coarse_accuracy=float(np.mean(pred == coarse)),
                macro_subtype_accuracy=float(np.mean([r['accuracy'] for r in rows if r['support']])),
                by_subtype=rows)

In [ ]:
# Validation check: arbitrarily ordered real rows, deliberately wrong on 5 apple examples (fine ID 0).
rows = validation_indices[::-1]
truth = train_coarse[rows]
witness = np.flatnonzero(train_fine[rows] == 0)[:5]
pred = truth.copy()
wrong_id = int((fine_to_coarse[0] + 1) % 20)
pred[witness] = wrong_id
val_check = evaluate_subtypes(pred, split='validation', row_ids=rows, manifest=MANIFEST)
assert val_check['n'] == 5000 and val_check['correct'] == 4995
assert val_check['coarse_accuracy'] == 4995 / 5000
assert np.isclose(val_check['macro_subtype_accuracy'], 4995 / 5000)
assert len(val_check['by_subtype']) == 100
assert val_check['by_subtype'][0]['support'] == 50 and val_check['by_subtype'][0]['correct'] == 45
assert val_check['by_subtype'][0]['wrong_coarse_counts'] == {wrong_id: 5}
assert all(r['accuracy'] == 1.0 for r in val_check['by_subtype'][1:])
assert evaluate_subtypes(np.eye(20)[pred], split='validation', row_ids=rows, manifest=MANIFEST) == val_check

test_check = evaluate_subtypes(test_coarse, split='test', row_ids=np.arange(10_000), manifest=MANIFEST)
assert test_check['correct'] == 10_000 and all(r['support'] == 100 for r in test_check['by_subtype'])

bad_rows = rows.copy()
bad_rows[0] = bad_rows[1]
bad_ids = pred.copy()
bad_ids[0] = 20
bad_scores = np.eye(20)[pred]
bad_scores[0, 0] = np.nan
for bad_pred, row_ids in [(pred, bad_rows), (bad_ids, rows), (bad_scores, rows), (pred[:-1], rows)]:
    try:
        evaluate_subtypes(bad_pred, split='validation', row_ids=row_ids, manifest=MANIFEST)
    except ValueError:
        pass
    else:
        raise AssertionError('Evaluator did not reject invalid inputs')
print('Subtype helper passed validation: multiple prediction formats, deliberate errors, arbitrary row order, and invalid-input rejection.')

## 6. Write the report
The Markdown report includes the verified summary and complete mapping. The JSON contains checksums, split settings, checks, and full count arrays.

In [ ]:
summary = {
    'dataset': 'CIFAR-100 Python archive',
    'execution': {'device': 'CPU', 'model_trained': False, 'python': platform.python_version(), 'numpy': np.__version__},
    'provenance': {'archive_origin': archive_origin, 'archive_bytes': archive_path.stat().st_size,
                   'archive_hashes': ARCHIVE_HASH, 'member_md5': member_hashes, 'references': REFERENCES},
    'taxonomy': {'fine_classes': len(fine_names), 'coarse_classes': len(coarse_names),
                 'fine_subtypes_per_coarse': np.bincount(fine_to_coarse).tolist()},
    'split': {'seed': SEED, 'validation_per_fine': VAL_PER_FINE,
              'algorithm': 'Within each fine class, rank official train rows by SHA-256 of ASCII cifar100-val-v1:<seed>:<row_id>, then row ID; hold out the first 50. Sort the resulting partitions by row ID.',
              'index_hash_encoding': 'sorted little-endian signed int64 bytes (no header)',
              'fit_indices_sha256': index_sha256(fit_indices),
              'validation_indices_sha256': index_sha256(validation_indices)},
    'distributions': {name: {'rows': int(f.sum()), 'by_fine_id': f.tolist(), 'by_coarse_id': c.tolist()}
                      for name, (f, c) in counts.items()},
    'checks': {'archive_and_members_verified': True, 'official_pair_mapping_agrees': True,
               'train_validation_disjoint_and_exhaustive': True, 'same_seed_repeats': True,
               'changed_seed_changes_membership': True, 'test_excluded_from_split': True,
               'subtype_helper_self_test_passed': True},
    'helper_self_test': {'constructed_predictions_not_model_results': True,
                         'validation_correct': val_check['correct'], 'validation_support': val_check['n'],
                         'validation_apple': val_check['by_subtype'][0], 'test_correct': test_check['correct'],
                         'test_support': test_check['n']},
}
(OUT / 'audit_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')

report = [
    '# CIFAR-100 coarse-label audit', '',
    '**Result: passed.** Verified all 60,000 image-aligned fine/coarse label pairs from the original Python archive. '
    'The official training and test sets independently yield the same mapping: '
    '100 fine classes, one parent per fine class, and exactly 5 fine subtypes for each of 20 coarse classes.', '',
    '| Partition | Images | Per fine subtype | Per coarse class |',
    '|---|---:|---:|---:|',
]
for name, (fine_counts, coarse_counts) in counts.items():
    report.append(f'| {name.replace("_", " ")} | {fine_counts.sum():,} | {fine_counts[0]:,} | {coarse_counts[0]:,} |')
report += [
    '', '## Split and provenance', '',
    f'- **Seed:** {SEED}. {summary["split"]["algorithm"]}',
    '- Fit and validation are disjoint and cover every original train row exactly once. '
    'Same-seed regeneration matches exactly; a different seed changes membership. '
    'Test is retained as the original official partition and has a separate row-ID namespace.',
    '- Index hashes use sorted little-endian signed int64 bytes, without a header.',
    f'  - Fit SHA-256: `{summary["split"]["fit_indices_sha256"]}`',
    f'  - Validation SHA-256: `{summary["split"]["validation_indices_sha256"]}`',
    f'- Archive obtained from: {archive_origin}',
    f'- MD5 (`{ARCHIVE_HASH["md5"]}`) matches the [publisher]({REFERENCES["publisher"]}); '
    f'SHA-256 (`{ARCHIVE_HASH["sha256"]}`) matches [Keras]({REFERENCES["keras"]}). '
    f'All three member MD5s match [TorchVision]({REFERENCES["torchvision"]}).',
    f'- CPU only, Python {platform.python_version()}, NumPy {np.__version__}. No model or GPU was used.', '',
    '## Coarse class → fine subtypes (official IDs and names)', '',
    '| Coarse ID | Coarse name | Fine IDs and names |', '|---:|---|---|',
]
for cid, name in enumerate(coarse_names):
    subtypes = ', '.join(f'{fid}: `{fine_names[fid]}`' for fid in np.flatnonzero(fine_to_coarse == cid))
    report.append(f'| {cid} | `{name}` | {subtypes} |')
report += [
    '', '## Using the subtype evaluator', '',
    'Predict the 20 coarse classes only. Use fine labels for auditing, validation stratification and diagnostics, not for fitting.', '',
    '```python', 'from subtype_eval import evaluate_subtypes',
    'report = evaluate_subtypes(predicted_coarse_ids_or_scores, split="validation", row_ids=prediction_row_ids)', '```', '',
    '`row_ids` are the original 0-based indices, in prediction order. Use official train row IDs for validation '
    'and official test row IDs for test (`split="test"`). The helper rejects missing, duplicated or foreign rows. '
    'For an N×20 score array, columns must match the official coarse ID order shown above. '
    'Subtype accuracy measures whether examples of that subtype were assigned their true coarse parent.', '',
    f'Self-test (constructed predictions, not a model result): {val_check["correct"]:,}/{val_check["n"]:,} '
    f'validation predictions correct with all five injected errors localized to apple; '
    f'{test_check["correct"]:,}/{test_check["n"]:,} correct for the test identity check.', '',
    '## Machine-readable files', '',
    '- `audit_summary.json`: full counts, checksums, verification results and split configuration.',
    '- `fine_counts.csv` and `coarse_counts.csv`: full distributions and official name/ID mapping.',
    '- `audit_manifest.npz`: original-order label vectors, hierarchy and split indices (no images).',
]
(OUT / 'audit_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')
print('\n'.join(report[:11]))
print('Report:', OUT / 'audit_report.md')